# Demonstrating secure communication using ML-KEM
This notebook demonstrates how Alice and Bob can use ML-KEM to securely negotiate a shared secret key over an insecure channel, and then use that key to send a message.

In [4]:
import sys
import os
sys.path.append(os.path.abspath('/home/xilinx/jupyter_notebooks/EE712/Project/ml-kem'))

from mlkem.ml_kem import ML_KEM
from mlkem.parameter_set import ML_KEM_768
import hashlib

### Step 1: Alice generates a keypair
Alice uses KeyGen. She will keep the decapsulation key (`dk`) secret, and send the encapsulation key (`ek`) to Bob.

In [5]:
ml_kem = ML_KEM(ML_KEM_768)

# Alice generates her keys
ek, dk = ml_kem.key_gen()

print(f"Alice's Encapsulation Key (public) size: {len(ek)} bytes")
print(f"Alice's Decapsulation Key (private) size: {len(dk)} bytes")

C extensions not found, falling back to pure Python implementation.


Alice's Encapsulation Key (public) size: 1184 bytes
Alice's Decapsulation Key (private) size: 2400 bytes


### Step 2: Bob generates a shared secret and encapsulates it
Bob receives Alice's `ek` and runs Encaps. This outputs a shared secret key (`k_bob`) and a ciphertext (`c`). He will keep the key secret and send the ciphertext `c` to Alice.

In [6]:
# Bob runs encaps using Alice's public key
k_bob, c = ml_kem.encaps(ek)

print(f"Bob's Shared Secret size: {len(k_bob)} bytes")
print(f"Ciphertext (sent to Alice) size: {len(c)} bytes")

Bob's Shared Secret size: 32 bytes
Ciphertext (sent to Alice) size: 1088 bytes


### Step 3: Alice decapsulates the joint shared secret
Alice receives Bob's ciphertext `c`. She uses her secret `dk` to run Decaps and recover the key.

In [7]:
# Alice recovers the shared secret
k_alice = ml_kem.decaps(dk, c)

# Let's verify they both derived the exact same key
assert k_alice == k_bob, "Keys do not match!"
print("Success! Alice and Bob successfully established the same shared secret across an insecure channel.")
print(f"The key is: {k_alice.hex()}")

Success! Alice and Bob successfully established the same shared secret across an insecure channel.
The key is: d2f7f100f0a77bd07cd5889eea77795f3427c3cd26f59a166066c7382508b4ba


### Step 4: Bob and Alice begin secure communication
Now that Alice and Bob share `k_alice = k_bob`, they can use a symmetric cipher to encrypt communication between the two of them. Since standard Python doesn't naively contain AES, we will use a demonstration stream cipher built out of the `SHAKE-256` hashing function.

In [8]:
def stream_cipher_crypt(key: bytes, message: bytes) -> bytes:
    """
    A simple stream cipher. We use the shared secret as a seed for the SHAKE-256 function.
    We squeeze out exactly as many pseudorandom bytes as our message, and XOR them together.
    (Note: A real-world app would use securely seeded AES-GCM or ChaCha20-Poly1305 per message).
    """
    key_stream = hashlib.shake_256(key).digest(len(message))
    return bytes([m ^ k for m, k in zip(message, key_stream)])

# Bob wants to send a secret message to Alice
bobs_message = b"Hello Alice! ML-KEM successfully secured this channel. FIPS-203 is cool."
print(f"Bob's original message: {bobs_message.decode()}")

# Bob encrypts the message using his shared key
encrypted_message = stream_cipher_crypt(k_bob, bobs_message)
print(f"Encrypted bits sent over channel: {encrypted_message.hex()[:40]}...")

Bob's original message: Hello Alice! ML-KEM successfully secured this channel. FIPS-203 is cool.
Encrypted bits sent over channel: b210d3ab2f1789c19b1fea81655eb40f8c3918d1...


In [9]:
# Alice receives the encrypted bits. She decrypts it using her shared key
decrypted_message = stream_cipher_crypt(k_alice, encrypted_message)

print(f"Alice decrypted: {decrypted_message.decode()}")
assert decrypted_message == bobs_message
print("Message successfully sent and received!")

Alice decrypted: Hello Alice! ML-KEM successfully secured this channel. FIPS-203 is cool.
Message successfully sent and received!
